# Pipes - Despliegue

- Cargamos el modelo
- Cargamos los datos futuros
- Aplicar tuberia

In [1]:
!pip install streamlit

In [2]:
#Cargamos librerias principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
def preprocesar(X, variables):
    COLS_BOOLEANAS = [
        'GESTANTE', 'POBLACION_ A _CARGO_ ICBF', 'CONFLICTO_ CON_ PAREJA_ O _EX_ PAREJA',
        'ENFERMEDAD_ CRONICA_ DOLOROSA_ O_ DISCAPACITANTE', 'PROBLEMAS_ ECONOMICOS',
        'MUERTE_ DE_ UN_ FAMILIAR', 'PROBLEMAS_ JURIDICOS', 'SUICIDIO_ DE_ UN_ FAMILIAR',
        'MALTRATO_ FISICO_SICOLOGICO_SEXUAL', 'PROBLEMAS_ LABORALES', 'PROBLEMAS_ FAMILIARES',
        'CONSUMO_ DE_ SPA', 'ANTECEDENTES_ FAMILIARES_ DE_ CONDUCTA_ SUICIDA',
        'IDEACION_ SUICIDA_ PERSISTENTE_', 'PLAN_ ORGANIZADO_ DE _SUICIDIO',
        'TRASTORNO_ DEPRESIVO', 'TRASTORNO_ DE_ PERSONALIDAD', 'TRASTORNO_ BIPOLAR',
        'ESQUIZOFRENIA', 'ANTECEDENTE_ VIOLENCIA _O_ ABUSO', 'ABUSO_ DE_  ALCOHOL',
        'AHORCAMIENTO', 'ARMA_ CORTOPUNZANTE', 'LANZAMIENTO_ AL_ VACIO',
        'LANZAMIENTO_ A_ VEHICULO', 'INTOXICACIONES',
    ]
    COLS_MULTICAT     = ['SEGURIDAD_ SOCIAL', 'ESTRATO_SOCIOECONOMICO', 'ESTADO_CIVIL', 'ESCOLARIDAD']
    COLS_BINARIAS_CAT = ['SEXO', 'AREA_ DE_ RESIDENCIA']

    df = X.copy()

    # Booleanas → get_dummies con drop_first=True genera columna _SI
    cols_bool = [c for c in COLS_BOOLEANAS if c in df.columns]
    if cols_bool:
        df = pd.get_dummies(df, columns=cols_bool, drop_first=True, dtype=int)

    # Multicategoría → get_dummies con drop_first=False
    cols_mc = [c for c in COLS_MULTICAT if c in df.columns]
    if cols_mc:
        df = pd.get_dummies(df, columns=cols_mc, drop_first=False, dtype=int)

    # Binarias → get_dummies con drop_first=True
    cols_bc = [c for c in COLS_BINARIAS_CAT if c in df.columns]
    if cols_bc:
        df = pd.get_dummies(df, columns=cols_bc, drop_first=True, dtype=int)

    return df.fillna(0).reindex(columns=variables, fill_value=0)

In [4]:
#Cargamos el pipeline con el modelo
import pickle
filename = 'modelo_reincidencia.pkl'
modelo, labelencoder, variables = pickle.load(open(filename, 'rb'))
modelo

LogisticRegression(C=np.float64(0.13067666894624388), class_weight='balanced',
                   max_iter=348, random_state=42)

In [5]:
#Cargamos los datos futuros
data = pd.read_csv("datos_futuros_reincidencia.csv")
data.head()

,SEXO,AREA_ DE_ RESIDENCIA,SEGURIDAD_ SOCIAL,ESTRATO_SOCIOECONOMICO,GESTANTE,POBLACION_ A _CARGO_ ICBF,ESTADO_CIVIL,ESCOLARIDAD,CONFLICTO_ CON_ PAREJA_ O _EX_ PAREJA,ENFERMEDAD_ CRONICA_ DOLOROSA_ O_ DISCAPACITANTE,...,TRASTORNO_ DE_ PERSONALIDAD,TRASTORNO_ BIPOLAR,ESQUIZOFRENIA,ANTECEDENTE_ VIOLENCIA _O_ ABUSO,ABUSO_ DE_ ALCOHOL,AHORCAMIENTO,ARMA_ CORTOPUNZANTE,LANZAMIENTO_ AL_ VACIO,LANZAMIENTO_ A_ VEHICULO,INTOXICACIONES
0,FEMENINO,CABECERA MUNICIPAL,SUBSIDIADO,2,NO,NO,SOLTERO/A,BASICA SECUNDARIA,SI,NO,...,NO,NO,NO,SI,NO,NO,NO,NO,NO,SI
1,MASCULINO,CABECERA MUNICIPAL,CONTRIBUTIVO,4,NO,NO,CASADO/A,PROFESIONAL,NO,NO,...,NO,NO,NO,NO,SI,NO,NO,NO,NO,SI
2,FEMENINO,RURAL DISPERSO,SUBSIDIADO,1,NO,NO,UNION LIBRE,BASICA PRIMARIA,SI,NO,...,NO,NO,NO,SI,NO,SI,NO,NO,NO,NO
3,MASCULINO,CABECERA MUNICIPAL,CONTRIBUTIVO,3,NO,NO,SOLTERO/A,TECNOLOGIA O TECNICA,NO,NO,...,NO,NO,NO,NO,SI,NO,SI,NO,NO,NO
4,FEMENINO,CABECERA MUNICIPAL,SUBSIDIADO,2,NO,NO,SOLTERO/A,BASICA SECUNDARIA,NO,NO,...,NO,NO,NO,NO,NO,NO,NO,NO,NO,SI


In [6]:
import streamlit as st

st.title('Despliegue de Modelo de Reincidencia')

st.write('Datos futuros cargados:')
st.dataframe(data)

# Preprocesar los datos
X_processed = preprocesar(data.copy(), variables)

# Hacer predicciones
Y_pred = modelo.predict(X_processed)
predictions = labelencoder.inverse_transform(Y_pred)

data['Prediccion'] = predictions

st.write('Resultados de la predicción:')
st.dataframe(data[['SEXO', 'AREA_ DE_ RESIDENCIA', 'Prediccion']])

st.success('¡Predicciones generadas con éxito!')

2026-05-28 22:09:52.606 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-28 22:09:53.051 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-05-28 22:09:53.053 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-28 22:09:53.054 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-28 22:09:53.056 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-28 22:09:53.058 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-28 22:09:53.058 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-28 22:09:53.073 Thread 'MainThread': mi

DeltaGenerator()

In [7]:
#Hacemos la prediccion con el modelo
X = preprocesar(data, variables)
Y_pred = modelo.predict(X)
print(labelencoder.inverse_transform(Y_pred))

['SI' 'NO' 'SI' 'SI' 'NO' 'SI' 'SI' 'SI' 'NO' 'SI' 'SI']


In [8]:
data['Prediccion'] = labelencoder.inverse_transform(Y_pred)
data

,SEXO,AREA_ DE_ RESIDENCIA,SEGURIDAD_ SOCIAL,ESTRATO_SOCIOECONOMICO,GESTANTE,POBLACION_ A _CARGO_ ICBF,ESTADO_CIVIL,ESCOLARIDAD,CONFLICTO_ CON_ PAREJA_ O _EX_ PAREJA,ENFERMEDAD_ CRONICA_ DOLOROSA_ O_ DISCAPACITANTE,...,TRASTORNO_ BIPOLAR,ESQUIZOFRENIA,ANTECEDENTE_ VIOLENCIA _O_ ABUSO,ABUSO_ DE_ ALCOHOL,AHORCAMIENTO,ARMA_ CORTOPUNZANTE,LANZAMIENTO_ AL_ VACIO,LANZAMIENTO_ A_ VEHICULO,INTOXICACIONES,Prediccion
0,FEMENINO,CABECERA MUNICIPAL,SUBSIDIADO,2,NO,NO,SOLTERO/A,BASICA SECUNDARIA,SI,NO,...,NO,NO,SI,NO,NO,NO,NO,NO,SI,SI
1,MASCULINO,CABECERA MUNICIPAL,CONTRIBUTIVO,4,NO,NO,CASADO/A,PROFESIONAL,NO,NO,...,NO,NO,NO,SI,NO,NO,NO,NO,SI,NO
2,FEMENINO,RURAL DISPERSO,SUBSIDIADO,1,NO,NO,UNION LIBRE,BASICA PRIMARIA,SI,NO,...,NO,NO,SI,NO,SI,NO,NO,NO,NO,SI
3,MASCULINO,CABECERA MUNICIPAL,CONTRIBUTIVO,3,NO,NO,SOLTERO/A,TECNOLOGIA O TECNICA,NO,NO,...,NO,NO,NO,SI,NO,SI,NO,NO,NO,SI
4,FEMENINO,CABECERA MUNICIPAL,SUBSIDIADO,2,NO,NO,SOLTERO/A,BASICA SECUNDARIA,NO,NO,...,NO,NO,NO,NO,NO,NO,NO,NO,SI,NO
5,FEMENINO,CABECERA MUNICIPAL,CONTRIBUTIVO,5,NO,NO,CASADO/A,MAESTRIA,SI,SI,...,SI,NO,NO,NO,NO,NO,NO,NO,SI,SI
6,MASCULINO,RURAL DISPERSO,SUBSIDIADO,1,NO,NO,SOLTERO/A,NINGUNO,NO,NO,...,NO,NO,SI,SI,SI,NO,NO,NO,NO,SI
7,FEMENINO,CABECERA MUNICIPAL,EXCEPCION,3,SI,NO,UNION LIBRE,BASICA SECUNDARIA,SI,NO,...,NO,NO,SI,NO,NO,NO,NO,NO,SI,SI
8,MASCULINO,CABECERA MUNICIPAL,CONTRIBUTIVO,4,NO,NO,DIVORCIADO/A,PROFESIONAL,SI,SI,...,NO,NO,NO,NO,NO,NO,SI,NO,NO,NO
9,FEMENINO,CABECERA MUNICIPAL,SUBSIDIADO,2,NO,NO,SOLTERO/A,BASICA SECUNDARIA,NO,NO,...,NO,NO,SI,NO,NO,NO,NO,NO,SI,SI


In [ ]:
!streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py



2026-05-28 22:10:24.549 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.237.75.77:8501



In [ ]:
!streamlit run app.py &>/content/logs.txt &